# Reproduce the three selected FDS submissions

This notebook clones the public GitHub repository, attaches to the FDS competition data, reconstructs the model-specific features, and writes the three selected submission files. The saved estimators were fitted on all 10,000 labeled training rows.

**Competition constraints:** test labels are never read; battle_id is only an output key; each model sees only the first 30 timeline snapshots. No external battle data or type chart is used.

The repository URL is configured below. Before running, attach the competition data through Kaggle's Input panel. Internet must be enabled for the GitHub clone and package install.

## Selected recipes

| Output folder | Ensemble | OOF recipe selected | User-reported leaderboard score |
|---|---|---|---:|
| c09 | Equal thirds: CatBoost, XGBoost, Logistic Regression | C09, 13 added features | 0.69333 |
| v334_highest_mean | Equal thirds: CatBoost, XGBoost, Logistic Regression | CatBoost F02/F04/N02/N03/N04; XGBoost C09; LR F02/N03 | 0.69266 |
| v334_r_both | Equal thirds: CatBoost, XGBoost, Logistic Regression | CatBoost F02/F04/N02/N03/N04; XGBoost F02/N01/N02/N03; LR C09 | 0.68933 (latest project registry) |

The latest project registry assigns 0.68933 to V334 R_BOTH and 0.698 to the different V337 F06 P1+P2 recipe. This notebook reproduces the requested V334 R_BOTH file, identified by its exact CSV hash. Scores are user-reported; the original selection label is preserved separately in the manifest.

In [ ]:
# Public GitHub repository cloned by this notebook.
REPOSITORY_URL = "https://github.com/MatteoAdilardi/FDS_pokemon_challenge_2026.git"

from pathlib import Path
import subprocess

REPO_DIR = Path("/kaggle/working/fds_selected_submissions")
if not REPOSITORY_URL.startswith("https://github.com/"):
    raise ValueError("Set REPOSITORY_URL to the public HTTPS GitHub clone URL.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / "run_submissions.py").is_file():
    raise FileNotFoundError("The cloned repository does not contain run_submissions.py at its root.")
print(f"Repository ready: {REPO_DIR}")

## Install the recorded Python environment

The requirements pin the numerical and model libraries used to create and verify these bundles. Kaggle also offers Dependency Manager for packages that must be available when a committed competition notebook runs without Internet.

In [ ]:
# Install the repository's pinned dependencies in this Kaggle session.
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 or newer is required by the pinned packages.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
     "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)

## Locate the attached challenge data

The runner requires train.jsonl, test.jsonl, and sample_submission.csv. Training labels are used only for optional refitting; the default path loads the saved full-train estimators and uses training rows only for target-free feature lookups and encoding.

In [ ]:
# Find the mounted competition data without assuming a specific Kaggle folder name.
input_root = Path("/kaggle/input")
data_matches = [p.parent for p in input_root.rglob("train.jsonl")
                if (p.parent / "test.jsonl").is_file()
                and (p.parent / "sample_submission.csv").is_file()]
if len(data_matches) != 1:
    raise RuntimeError(f"Expected one attached competition data directory; found {data_matches}")
DATA_DIR = data_matches[0]
OUTPUT_DIR = Path("/kaggle/working/submissions")
REFIT_MODELS = False  # Keep False to reproduce the exact frozen full-train artifacts.
print(f"Competition data: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Generate and verify all three CSV files

The default uses the serialized CatBoost, XGBoost, and Logistic Regression estimators already refitted on all 10,000 labeled rows. Set REFIT_MODELS = True only if you want to execute a fresh full-training fit; that mode reports whether its output still matches the frozen submission hash.

In [ ]:
# Run the same prediction pipeline for each selected recipe.
command = [
    sys.executable, str(REPO_DIR / "run_submissions.py"),
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
]
if REFIT_MODELS:
    command.append("--refit-models")
subprocess.run(command, cwd=REPO_DIR, check=True)

## Review outputs

Each recipe has its own submission.csv, component_predictions.csv, and metadata.json. Download only the submission.csv for the recipe you intend to submit. The component file is diagnostic and is not accepted by Kaggle's submission interface.

In [ ]:
# Summarize generated submission hashes and row counts.
import hashlib
import pandas as pd

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

summary = []
for recipe in ("c09", "v334_highest_mean", "v334_r_both"):
    submission_path = OUTPUT_DIR / recipe / "submission.csv"
    frame = pd.read_csv(submission_path)
    summary.append({
        "recipe": recipe,
        "rows": len(frame),
        "columns": ",".join(frame.columns),
        "sha256": file_sha256(submission_path),
        "submission_path": str(submission_path),
    })
display(pd.DataFrame(summary))